<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.5-langgraph-brain/notebooks/GCP_Capstone_8.5_LangGraph.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.5 Construct DocuMind's LangGraph Brain — StateGraph, Checkpointers & a Summarise Node, on the Lane
**Netsetos GenAI Engineering — GCP Capstone** · Module 8 · rebuilt on the live lane, 8 September 2026

Lesson 6.2 wrote the tool loop by hand, 6.4 got it back from a framework, 8.1 built it on ADK. All three keep the conversation in a Python list that dies with the process. This notebook makes the loop a **graph** and the graph **durable**, over the kit's one `retrieve()`:

- `MessagesState` as a **reducer**, not a variable
- a `StateGraph` with **one router**, and why that beats two conditional edges
- `ToolNode` over an adapter whose tenant comes from the **thread id**
- `InMemorySaver` (what the demo project's chat service runs on) -> `SqliteSaver` -> `AsyncPostgresSaver` (the full profile), and what each loses
- a **summarise node** with `RemoveMessage`, and the two ways to corrupt a history
- `stream_mode='messages'` for a chat UI

*Every question is a golden row of 4.8's set. API facts verified 2026-09-04 by running the graph, against langgraph 1.2.11.*


## Setup
7.1's: the kit cloned and imported, the identity minted per call as `documind-ui-sa`. The history budget is deliberately small so the summarise node fires in this notebook.


In [ ]:
!pip install -q "langgraph==1.2.11" "langgraph-checkpoint-sqlite==3.1.1" "langgraph-checkpoint-postgres==3.1.2" \
                "psycopg[binary]==3.3.5" "langchain-google-genai==4.4.0" google-genai==2.22.0 google-auth==2.57.1 requests==2.34.2

# psycopg[binary] is NOT optional, and the failure is opaque: the Postgres checkpointer imports
# psycopg, psycopg needs libpq, and without the binary wheel the ImportError says "no pq wrapper
# available" and never mentions Postgres or LangGraph. Reproduced 2026-09-04.

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every agent in this module imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": f"https://documind-api-{NUMBER}.{REGION}.run.app",
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - 8.7's gate fails a paste.
# Deliberately small so the summarise node ACTUALLY FIRES inside this notebook. A budget you never
# cross teaches nothing: you would read the node, believe it, and never watch it run.
HISTORY_BUDGET = 6                       # messages, not tokens - see Cell 10
KEEP_RECENT = 4                          # how many messages survive a summarisation
print("kit:", KIT, "| API:", os.environ["RAG_API_URL"])


## Cell 1: State is a reducer, not a variable
The one idea that makes everything below behave strangely if you skip it. A reduced field is a ledger: return `{'messages': [m]}` and m is **appended**.


In [ ]:
from langgraph.graph import MessagesState

# MessagesState is one field, `messages`, with a REDUCER attached. That word is the whole lesson.
#
# A normal state field is a variable: return {"x": 3} and x becomes 3.
# A reduced field is a fold: return {"messages": [m]} and m is APPENDED to what was there.
#
# So a node never has to read the history to add to it, two nodes writing messages in the same
# step do not clobber each other, and - the part that matters in Cell 10 - deletion has to be an
# explicit instruction rather than "assign a shorter list".
#
# You can define your own state when you need more than messages:
#
#     from typing import Annotated, TypedDict
#     from langgraph.graph.message import add_messages
#
#     class DocuMindState(TypedDict):
#         messages: Annotated[list, add_messages]   # the reducer, spelled out
#         tenant_id: str                            # plain field: last write wins
#         retrieved_chunk_ids: list[str]
#
# This lesson stays on MessagesState because the extra fields would be scenery. Reach for a custom
# state the moment a node needs to pass something that is not a message.


## Cell 2: The tool
An adapter over the kit's one `retrieve()`. The tenant comes from the thread id the checkpointer is keyed on; the model sees `query` and nothing else.


In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool


@tool
def retrieve(query: str, config: RunnableConfig) -> dict:
    """Retrieve grounded passages from DocuMind's document corpus, with the lane's cited answer.

    Args:
        query: The question, in natural language.
    """
    # THE TENANT COMES FROM THE THREAD, not from a module global and never from the model. LangChain
    # injects the RunnableConfig, so the tool reads the same thread_id the checkpointer is keyed on
    # (Cell 6) - which means one process serving three tenants cannot answer from the wrong corpus.
    # The body DELEGATES: this is an adapter over the kit's one retrieve(), not a second copy.
    tenant_id = config["configurable"]["thread_id"].split(":", 1)[0]
    return documind_tools.retrieve(query, tenant_id=tenant_id, top_k=5, brain="langgraph")


TOOLS = [retrieve]
print("model sees:", sorted(retrieve.tool_call_schema.model_json_schema()["properties"]))   # query only - no config, no tenant


## Cell 3: The nodes
`agent` calls the model. `summarise` collapses old history, and returns deletions alongside - the reducer in action.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, RemoveMessage

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    vertexai=True, project=PROJECT_ID, location="global",   # 3.x generation is global-only
    thinking_level="low",
).bind_tools(TOOLS)


def agent(state: MessagesState) -> dict:
    """One model call. Returns the reply to be APPENDED to the history."""
    return {"messages": [llm.invoke(state["messages"])]}


def _safe_cut(msgs: list, keep_recent: int) -> int:
    """Index to cut at: at least keep_recent survive, and NEVER mid-turn.

    Walk backwards from the naive boundary until the message there is a HumanMessage. That is
    the only position where no assistant message with tool_calls is separated from the
    ToolMessages answering it - the failure "keep the last N" commits routinely, because N lands
    wherever it lands.
    """
    i = max(0, len(msgs) - keep_recent)
    while i > 0 and type(msgs[i]).__name__ != "HumanMessage":
        i -= 1
    return i


def summarise(state: MessagesState) -> dict:
    """Collapse old history into one summary message. See Cell 10 for why RemoveMessage."""
    msgs = state["messages"]
    cut = _safe_cut(msgs, KEEP_RECENT)
    keep, drop = msgs[cut:], msgs[:cut]
    if not drop:
        return {"messages": []}
    text = llm.invoke(
        [{"role": "user",
          "content": "Summarise this conversation in under 120 words. Keep every clause code and figure "
                     "that was quoted; drop the pleasantries.\n\n"
                     + "\n".join(f"{type(m).__name__}: {m.content}" for m in drop)}]).content
    # Delete the dropped ids and append the summary. Nothing else.
    #
    # Do NOT try to "reorder" by removing the kept messages and re-adding them: a RemoveMessage
    # and a re-add of the same id in one update leaves the message DELETED, so the conversation
    # loses its most recent answer. Ordering is handled structurally instead - see the graph. This
    # node runs at the START of a turn, before the model is called, so the summary is appended
    # while the newest thing in the history is still the user's previous answer.
    return {"messages": [RemoveMessage(id=m.id) for m in drop]
                        + [AIMessage(content=f"[earlier conversation] {text}")]}


## Cell 4: One router
Order here is correctness, not style: a pending tool call must be answered even when the conversation is over budget, so summarising happens between turns.


In [ ]:
from typing import Literal


def entry(state: MessagesState) -> Literal["summarise", "agent"]:
    """Runs FIRST, before the model. Trim the history, then answer.

    Putting the budget check here rather than after the model call is what makes the "tools
    win" problem impossible instead of merely unlikely: summarisation happens BETWEEN turns,
    when there is never a pending tool call to orphan. It also keeps the ordering right - the
    summary is appended while the newest message is still the previous answer, so this turn's
    reply lands after it.
    """
    return "summarise" if len(state["messages"]) > HISTORY_BUDGET else "agent"


def after_agent(state: MessagesState) -> Literal["tools", "__end__"]:
    """Runs after the model. Only one question left: did it ask for a tool?"""
    return "tools" if getattr(state["messages"][-1], "tool_calls", None) else "__end__"


# WHY TWO SMALL ROUTERS AND NOT TWO CONDITIONAL EDGES ON ONE NODE.
# LangGraph lets you attach several conditional edges to the same node, and it evaluates ALL of
# them. Written that way - add_conditional_edges("agent", tools_condition) plus
# add_conditional_edges("agent", needs_summary) - you do not get a choice between two
# destinations. You get FAN-OUT: when both conditions return a real node, both nodes run in the
# same superstep. The graph would summarise the history CONCURRENTLY with executing the tool call.
# Registration order does not save you. One function returning one destination is the only
# version where "either/or" is what actually happens.


## Cell 5: The graph
Four edges, and you have the whole control flow: data you can draw, diff, and snapshot after every step.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode


def build_graph(checkpointer):
    """The DocuMind brain. Same graph on every profile; only the checkpointer changes."""
    g = StateGraph(MessagesState)

    g.add_node("agent", agent)
    g.add_node("tools", ToolNode(TOOLS))     # runs the calls, appends one ToolMessage each
    g.add_node("summarise", summarise)

    g.add_conditional_edges(START, entry)           # -> summarise | agent   (before the model)
    g.add_edge("summarise", "agent")                # trim, THEN answer
    g.add_conditional_edges("agent", after_agent)   # -> tools | END
    g.add_edge("tools", "agent")                    # tool results go back for another model call

    return g.compile(checkpointer=checkpointer)


# Read the four edges and you have the whole control flow, which is the thing a while-loop
# cannot give you: this is DATA. You can draw it, diff it in review, and - the reason this
# lesson exists - snapshot it after every step.
for e in build_graph(None).get_graph().edges:
    print(f"  {e.source:>9} -> {e.target:<9}{'  (conditional)' if e.conditional else ''}")


## Cell 6: The thread id is the tenancy boundary
Everything the checkpointer stores hangs off this string, and the tool reads the tenant back out of it.


In [ ]:
# The thread id IS the tenancy boundary. Everything the checkpointer stores hangs off it.
#
# tenant : user : session
#   tenant  - never from the question, never from the model. From the verified identity (12.8).
#   user    - so one tenant's users cannot read each other's threads.
#   session - so "new chat" is a new thread rather than a wiped one.
#
# Get this wrong and the bug is not an error, it is a stranger's conversation appearing in
# someone's window. There is no exception to catch. deploy/services/chat/agent.py builds this
# string from the roster lookup, and the retrieve adapter above reads the tenant back out of it.
def thread_config(tenant_id: str, user_id: str, session_id: str) -> dict:
    parts = (tenant_id, user_id, session_id)
    if not all(parts):
        raise ValueError("tenant, user and session ids must all be non-empty")
    if any(":" in p for p in parts):
        # The tenant is the FIRST segment, so a colon in a user id cannot forge a different tenant -
        # the string still begins with "{tenant}:". What it can do is collide with another user INSIDE
        # your tenant: user "a:b" + session "c" equals user "a" + session "b:c". Ban the separator.
        raise ValueError("tenant, user and session ids must not contain ':'")
    return {"configurable": {"thread_id": f"{tenant_id}:{user_id}:{session_id}"}}


CONFIG = thread_config(TENANT, "priya", "sess-1")
print(CONFIG)


## Cell 7: InMemorySaver - it works, and it is wrong to ship
Watch the memory work first. The defect is in the lifetime, which is invisible in a notebook - and it is what the demo project's chat service runs on, deliberately.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

app = build_graph(InMemorySaver())

# THREE questions, and the third is the point. HISTORY_BUDGET is 6 and a turn adds about four
# messages, so the third turn STARTS over budget and you get to watch the summarise node run.
# Three golden rows: lk-06 (60 days), lk-04 (15 days), jn-02 (unused leave cannot shorten notice).
for question in ["What is the notice period for a confirmed E3?",
                 "And during probation?",
                 "Can unused leave shorten the notice period?"]:
    result = app.invoke({"messages": [{"role": "user", "content": question}]}, CONFIG)
    print(f"\nQ: {question}")
    print(f"A: {result['messages'][-1].content[:220]}")

history = app.get_state(CONFIG).values["messages"]
print(f"\nhistory: {len(history)} messages")
print("summarised:", any("[earlier conversation]" in str(m.content) for m in history))
print("last message is the ANSWER, not the summary:", "[earlier conversation]" not in str(history[-1].content))
assert any(getattr(m, "tool_calls", None) for m in history) or any(type(m).__name__ == "ToolMessage" for m in history), "no tool was ever called"

# It works, and it is the wrong thing to ship.
#
# InMemorySaver holds checkpoints in a Python dict inside ONE process. On Cloud Run that means:
#   - two instances hold two different conversations, and which one you get depends on routing
#   - scale-to-zero deletes every conversation in progress
#   - a deploy deletes them again
# Nothing errors. Users simply find the assistant has forgotten them, intermittently, and you
# cannot reproduce it locally because locally there is one process. The demo project's chat
# service runs on exactly this - CHECKPOINT_DSN=memory, which agent.py logs as "tests only" -
# because a demo can live with it and a product cannot.


### SqliteSaver: prove it survives a restart
The claim is durability, so the demo has to kill something. The saver is discarded between the two blocks and a new one opened on the same file.


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

# A file on disk. Survives a kernel restart, which is exactly the property we want to SEE.
with SqliteSaver.from_conn_string("documind_threads.db") as saver:
    app = build_graph(saver)
    app.invoke({"messages": [{"role": "user", "content": "What is the notice period for a confirmed E3?"}]}, CONFIG)
    print("turn 1 stored")

# Now throw the saver away. This is what a restart does - the object is gone, the file is not.
with SqliteSaver.from_conn_string("documind_threads.db") as saver:
    app = build_graph(saver)
    recovered = app.get_state(CONFIG).values["messages"]
    print(f"recovered {len(recovered)} messages from the checkpoint")
    result = app.invoke({"messages": [{"role": "user", "content": "And during probation?"}]}, CONFIG)
    print("A:", result["messages"][-1].content[:220])

# Re-run this cell and the count climbs - 4, then 8, then 12. That is not a bug, it is the point:
# the file is not reset between runs, which is exactly the property a restart relies on. Delete
# documind_threads.db if you want a clean count.
#
# SqliteSaver is the DEVELOPMENT lane, for the same reason 6.4's Chroma directory was: a file on a
# Cloud Run instance is per-instance and evaporates on deploy. Perfect for a laptop, wrong for a service.


## Cell 8: AsyncPostgresSaver on Cloud SQL - the full profile
`setup()` once, from a migration job. Cloud Run reaches Cloud SQL over a unix socket. The kit's `chat/agent.py` is this cell shipped.


In [ ]:
# THE PRODUCTION LANE - the full profile's. Same graph, a database-backed saver.
from contextlib import asynccontextmanager
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

# The DSN is a SECRET. It carries a password and it reaches Cloud SQL. It belongs in Secret
# Manager, mounted as an env var by the service - never in the image, never in the repo. The kit's
# terraform/cloudsql.tf writes it and commands/lesson-12.8.sh mounts it with --set-secrets; on the
# lean profile there is no Cloud SQL and this string is "memory" instead. Empty here on purpose:
# this cell DEFINES the production factory without trying to connect from a notebook.
DSN = os.environ.get("CHECKPOINT_DSN", "")   # postgresql://u:pw@/db?host=/cloudsql/PROJ:REGION:INST


# THE TRAP IN THIS CELL, and it is the one everybody hits:
#
#     async with AsyncPostgresSaver.from_conn_string(DSN) as saver:
#         return build_graph(saver)          # <-- WRONG
#
# from_conn_string is an async context manager. Returning from inside it runs __aexit__ and
# CLOSES the connection, so the graph you just returned is holding a dead handle and every
# request fails. The saver has to outlive the function that made it: a long-lived pool, opened
# once at startup and closed at shutdown. deploy/services/chat/agent.py does it with an ExitStack
# in the FastAPI lifespan - read build_checkpointer() there; it is this cell, shipped.
@asynccontextmanager
async def production_app():
    """Yields a graph whose connection stays open for the lifetime of the process."""
    if not DSN or DSN == "memory":
        raise RuntimeError("CHECKPOINT_DSN names no database - the full profile's cloudsql.tf creates one")
    async with AsyncPostgresSaver.from_conn_string(DSN) as saver:
        yield build_graph(saver)             # yield, not return: the CM stays open around it


# setup() creates the tables and is idempotent, which makes it tempting to call on startup. Do
# not. Its migrations take ACCESS EXCLUSIVE locks and insert into a primary-keyed
# checkpoint_migrations table, so N instances racing on a scale-out produce a deploy that fails
# at random and succeeds on retry. Run it ONCE, from a migration job - the kit's migrate.py, run
# by the `documind-checkpoint-setup` Cloud Run job in lesson-12.8.sh.
#
# Two connection notes: psycopg needs libpq (install psycopg[binary]); and Cloud Run reaches
# Cloud SQL over a UNIX SOCKET, which is why the DSN carries host=/cloudsql/PROJECT:REGION:INSTANCE.
print("production factory defined; DSN set:", bool(DSN and DSN != "memory"))


## Cell 9: Never let a user reach `list(filter=)`
`saver.list()` matches `filter` against checkpoint metadata across the table. The server builds the scope from the verified identity.


In [ ]:
# The checkpointer can enumerate threads. Read the signature before you expose it.
#
#     saver.list(config, *, filter=None, before=None, limit=None)
#
# `filter` is a plain dict matched against checkpoint METADATA, scoped to nothing except the
# config you pass. So:
#
#     threads = saver.list(None, filter=request.json["filter"])     # NEVER
#
# is an admin endpoint wearing a user's clothes. The safe shape: the SERVER builds the scope from
# the verified identity, and the user's input never reaches `filter`.
def list_user_threads(saver, tenant_id: str, user_id: str, limit: int = 20):
    """Only ever the caller's own THREADS. tenant_id and user_id come from the verified identity.

    list() returns CHECKPOINTS, not threads - one row per saved step. A single busy conversation
    produces hundreds, so a naive page of 100 can surface exactly one thread. De-duplicate on
    thread_id and keep reading until you have enough DISTINCT ones.
    """
    prefix = f"{tenant_id}:{user_id}:"
    seen, out = set(), []
    for cp in saver.list(None):                      # newest first
        tid = str(cp.config["configurable"]["thread_id"])
        if not tid.startswith(prefix) or tid in seen:
            continue
        seen.add(tid)
        out.append(tid)
        if len(out) >= limit:
            break
    return out


with SqliteSaver.from_conn_string("documind_threads.db") as saver:
    print("priya's threads:", list_user_threads(saver, TENANT, "priya"))
    print("someone else's :", list_user_threads(saver, TENANT, "raj"))


## Cell 10: The summarise node, and two ways to corrupt a history


In [ ]:
# WHY RemoveMessage AND NOT "just return a shorter list".
#
# `messages` has a reducer. Returning {"messages": [a, b]} APPENDS a and b - it does not replace
# the history with them. There is no assignment to make. Deletion is an instruction:
#
#     RemoveMessage(id=m.id)                   # delete this one
#     RemoveMessage(id=REMOVE_ALL_MESSAGES)    # from langgraph.graph.message - clear the channel
#
# Two rules the budget has to respect, both of which produce a 400 from the model if broken:
#   1. Never drop an assistant message with tool_calls unless you also drop its ToolMessages.
#   2. Never leave a ToolMessage whose call you dropped, for the same reason.
# Keeping the last N messages can violate both. _safe_cut slices at a turn boundary instead.
#
# And the budget is in MESSAGES, which is a proxy. The thing you actually pay for is TOKENS
# (lesson 2.1). Messages are easy to count and roughly monotonic in tokens; when the two diverge
# - one enormous tool result - the proxy is what fails you. Count tokens once it matters.
with SqliteSaver.from_conn_string("documind_threads.db") as saver:
    app = build_graph(saver)
    for m in app.get_state(CONFIG).values["messages"]:
        print(f"  {type(m).__name__:<13} {str(m.content)[:90]!r}")


## Cell 11: Streaming


In [ ]:
# stream_mode="messages" yields (message_chunk, metadata) pairs as the model produces them -
# the mode you want behind a chat UI, because the user sees tokens rather than a spinner.
with SqliteSaver.from_conn_string("documind_threads.db") as saver:
    app = build_graph(saver)
    for chunk, meta in app.stream(
            {"messages": [{"role": "user", "content": "Summarise ACME's leave policy in three sentences, with clause codes."}]},
            thread_config(TENANT, "priya", "sess-2"),
            stream_mode="messages"):
        # FILTER. This mode yields chunks from EVERY node, tool results included - so a naive
        # print(chunk.content) puts raw JSON like {"citations": [...]} into the user's chat window
        # between sentences. Ask for the node you mean.
        if meta.get("langgraph_node") == "agent" and getattr(chunk, "content", None):
            print(chunk.content, end="", flush=True)
print()

# The other modes, and when each is the right one:
#   "values"   - the whole state after each step. Debugging.
#   "updates"  - only what each node returned. The cheapest useful trace.
#   "messages" - token-level. The UI.
#   "custom"   - whatever you emit from inside a node. Progress bars for slow tools.


## What ships

The graph is identical on every profile. **Only the checkpointer changes** - and `deploy/services/chat/agent.py` chooses it from the profile:

| Lane | Checkpointer | Survives | Where |
|---|---|---|---|
| unit test, and the demo project | `InMemorySaver` (`CHECKPOINT_DSN=memory`, logged as "tests only") | nothing, deliberately | lean profile |
| laptop | `SqliteSaver` | a kernel restart | `make chat-local` |
| production | `PostgresSaver` on Cloud SQL | a deploy, a scale-to-zero, an instance dying | full profile |

That is the same contract 6.4 drew around `DOCUMIND_PROFILE`: if a profile ever needs its own graph, the abstraction has failed and you are maintaining two applications under one name. 8.7 runs this graph as the `langgraph` brain of the chat service, beside three others.

## ✅ Lesson 8.5 complete
- ✅ The kit's retrieve() behind an adapter whose tenant is the thread id; the model sees only the query
- ✅ A reducer, one router, four edges; three golden rows through the graph, the third over budget
- ✅ InMemory (the demo's honest lane), Sqlite (a restart survived), Postgres (defined, and shipped in the kit)
- ✅ Threads listed only within the caller's own scope
- ✅ Streaming filtered to the agent node
